In [ ]:
from datetime import datetime
import glob
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()


# Define input (Silver) and output (Bronze) directory paths


# silver_dir = os.getenv("SILVERALL")
silver_dir = os.getenv("SILVERINC")
bronze_dir = os.getenv("BRONZEN")

# Update silver_dir to include a subfolder named with the current date (e.g., YYYY-MM-DD or YYYYMMDD)
current_date_folder = datetime.now().strftime("%Y-%m-%d")
silver_dir = os.path.join(silver_dir, current_date_folder)

# Create Bronze directory if it doesn't already exist
if not os.path.exists(bronze_dir):
    os.makedirs(bronze_dir)
    print(f"Created directory: {bronze_dir}")

# Find all Excel and CSV files inside the Silver folder
file_patterns = [
    os.path.join(silver_dir, "*.xlsx"),
    os.path.join(silver_dir, "*.xls"),
    os.path.join(silver_dir, "*.csv"),
]

all_files = []
for pattern in file_patterns:
    all_files.extend(glob.glob(pattern))

print(f"Found {len(all_files)} file(s) in Silver layer:")
for f in all_files:
    print(f" - {os.path.basename(f)}")

if not all_files:
    print("\n❌ No matching Excel or CSV files were found in the Silver folder.")
else:
    dataframes = []

    # Read each file and append to our list
    for file_path in all_files:
        try:
            filename = os.path.basename(file_path)
            if file_path.endswith(".csv"):
                df = pd.read_csv(file_path)
            else:
                df = pd.read_excel(file_path)

            # --- Drop existing S. No. columns if present ---
            sno_variations = [
                "S. No.",
                "S.No.",
                "S.No",
                "S. No",
                "SNo",
                "S_No",
                "s.no.",
                "s.no",
            ]
            cols_to_drop = [
                col for col in df.columns if str(col).strip() in sno_variations
            ]
            if cols_to_drop:
                df.drop(columns=cols_to_drop, inplace=True)

            # Optionally add a metadata column tracking the source filename
            df["Source_File"] = filename

            dataframes.append(df)
            print(f"✓ Successfully loaded: {filename} ({len(df)} rows)")
        except Exception as e:
            print(f"❌ Error reading {os.path.basename(file_path)}: {e}")

    # Merge/Concatenate all DataFrames into one
    if dataframes:
        merged_df = pd.concat(dataframes, ignore_index=True)

        # --- Generate new continuous S. No. column ---
        merged_df.insert(0, "S. No.", range(1, len(merged_df) + 1))

        # Define output destination file path
        timestamp = datetime.now().strftime("%Y%m%d")
        output_file_path = os.path.join(bronze_dir, f"{timestamp}.xlsx")

        # Save merged dataframe to Excel
        merged_df.to_excel(output_file_path, index=False)
        print(f"\n✅ Merge complete! Total combined rows: {len(merged_df)}")
        print(f"📁 Output file created at:\n   {output_file_path}")
    else:
        print("\n❌ Failed to parse any data from the identified files.")

Found 31 file(s) in Silver layer:
 - MOEFCC_20260730_204451.xlsx
 - SEIAA_ANDAMAN_AND_NICOBAR_ISLANDS_20260730_193931.xlsx
 - SEIAA_ANDHRA_PRADESH_20260730_150534.xlsx
 - SEIAA_ARUNACHAL_PRADESH_20260730_185609.xlsx
 - SEIAA_ASSAM_20260730_180013.xlsx
 - SEIAA_BIHAR_20260730_172330.xlsx
 - SEIAA_CHHATTISGARH_20260730_171230.xlsx
 - SEIAA_DELHI_20260730_192718.xlsx
 - SEIAA_GOA_20260730_182526.xlsx
 - SEIAA_GUJARAT_20260730_150711.xlsx
 - SEIAA_HARYANA_20260730_180505.xlsx
 - SEIAA_HIMACHAL_PRADESH_20260730_173435.xlsx
 - SEIAA_JAMMU_AND_KASHMIR_20260730_174444.xlsx
 - SEIAA_JHARKHAND_20260730_174901.xlsx
 - SEIAA_KARNATAKA_20260730_150745.xlsx
 - SEIAA_KERALA_20260730_182540.xlsx
 - SEIAA_MADHYA_PRADESH_20260730_172113.xlsx
 - SEIAA_MAHARASHTRA_20260730_172011.xlsx
 - SEIAA_MANIPUR_20260730_193740.xlsx
 - SEIAA_MEGHALAYA_20260730_175521.xlsx
 - SEIAA_ODISHA_20260730_150727.xlsx
 - SEIAA_PUDUCHERRY_20260730_193630.xlsx
 - SEIAA_PUNJAB_20260730_172258.xlsx
 - SEIAA_RAJASTHAN_20260730_153